In [1]:
!pip install xgboost lightgbm catboost imbalanced-learn shap optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 20.9 MB/s eta 0:00:00


# **Importing Libraries**

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

import joblib
import warnings
warnings.filterwarnings("ignore")

# **Loading datasets**

In [3]:
train = pd.read_csv("/content/UNSW_NB15_training-set(in).csv")
test = pd.read_csv("/content/UNSW_NB15_testing-set(in).csv")

print(train.shape)
print(test.shape)

train.head()

(175341, 45)
(82332, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


# **Data Preprocessing**

In [4]:
suspicious_classes = ['Analysis', 'Backdoor', 'Reconnaissance']

train['target'] = train['attack_cat'].apply(
    lambda x: 1 if x in suspicious_classes else 0
)

test['target'] = test['attack_cat'].apply(
    lambda x: 1 if x in suspicious_classes else 0
)

Keeping important features only and dropping features that are not required

In [5]:
drop_cols = ['id','label','attack_cat']

Create train/test

In [6]:
X_train = train.drop(columns=drop_cols + ['target'])
y_train = train['target']

X_test = test.drop(columns=drop_cols + ['target'])
y_test = test['target']

# **Encode Categories**

In [7]:
cat_cols = X_train.select_dtypes(include='object').columns

encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    combined = pd.concat([
        X_train[col].astype(str),
        X_test[col].astype(str)
    ])

    le.fit(combined)

    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    encoders[col] = le

# **Train Model**

In [8]:
imp = SimpleImputer(strategy='median')

X_train = pd.DataFrame(
    imp.fit_transform(X_train),
    columns=X_train.columns
)

X_test = pd.DataFrame(
    imp.transform(X_test),
    columns=X_test.columns
)

In [9]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(y_train.value_counts())
print(y_train_smote.value_counts())

target
0    161104
1     14237
Name: count, dtype: int64
target
0    161104
1    161104
Name: count, dtype: int64


In [10]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

In [11]:
lgbm = LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)

In [12]:
cat = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    verbose=0
)

In [13]:
stack_model = StackingClassifier(
    estimators=[
        ('xgb', xgb),
        ('lgbm', lgbm),
        ('cat', cat)
    ],
    final_estimator=LogisticRegression(),
    cv=5
)

In [ ]:
stack_model.fit(
    X_train_smote,
    y_train_smote
)

[LightGBM] [Info] Number of positive: 161104, number of negative: 161104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.071989 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9046
[LightGBM] [Info] Number of data points in the train set: 322208, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 128883, number of negative: 128883
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.054627 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9111
[LightGBM] [Info] Number of data points in the train set: 257766, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initsco

In [ ]:
prob = stack_model.predict_proba(X_test)[:,1]

In [ ]:
pred = (prob > 0.65).astype(int)

In [ ]:
print("Accuracy:", accuracy_score(y_test,pred))

print("F1:",f1_score(y_test,pred))

print(classification_report(y_test,pred))

print(confusion_matrix(y_test,pred))

In [ ]:
auc = roc_auc_score(y_test, prob)

print("AUC:", auc)

In [ ]:
joblib.dump(
    stack_model,
    "network_bouncer_model.pkl"
)

In [ ]:
import matplotlib.pyplot as plt

xgb.fit(X_train_smote, y_train_smote)
importance = xgb.feature_importances_

indices = np.argsort(importance)[::-1]

plt.figure(figsize=(12,8))

plt.bar(
    range(20),
    importance[indices[:20]]
)

plt.xticks(
    range(20),
    X_train.columns[indices[:20]],
    rotation=90
)

plt.show()

In [ ]:
def rule_based_detection(row):

    risk_score = 0

    if row['ct_src_dport_ltm'] > 30:
        risk_score += 3

    if row['ct_dst_ltm'] > 20:
        risk_score += 3

    if row['dur'] < 2:
        risk_score += 2

    if row['state'] in ['INT','RST','REQ']:
        risk_score += 2

    if row['dbytes'] < 50:
        risk_score += 1

    if row['service'] == '-':
        risk_score += 1

    if risk_score >= 8:
        return "HIGH_RISK"

    return "SEND_TO_ML"

In [ ]:
test["rule_output"] = test.apply(

    rule_based_detection,

    axis=1
)

In [ ]:
final_predictions = []

for index,row in test.iterrows():

    rule = rule_based_detection(row)

    if rule == "HIGH_RISK":

        final_predictions.append(1)

    else:

        data = X_test.loc[[index]]

        pred = stack_model.predict(data)[0]

        final_predictions.append(pred)

In [ ]:
print(

accuracy_score(

y_test,

final_predictions

))